# 03_gold
Implements the pre-registered forecast rule (docs/decision_log.md):
- `gold.forecast_rule`: the rule's parameters, one row per hindcast.
- `gold.forecast_vs_actual`: per article per week (8 before to 12 after the target event): forecast band from the analog vs actual views, band position, drift flag.
- `gold.hindcast_summary`: one row per article with the headline result.

In [0]:
%sql
SELECT topic, count(*) AS weeks, min(week_start) AS first_week, max(week_start) AS last_week,
       count_if(is_incomplete) AS incomplete_weeks
FROM nba_demand.silver.pageviews_weekly
GROUP BY topic;

In [0]:
%sql
CREATE OR REPLACE TABLE nba_demand.gold.forecast_rule
COMMENT 'Gold: parameters of the pre-registered forecast rule (see docs/decision_log.md). One row per hindcast.'
AS SELECT
  'olympic_finals_2024_to_nba_paris_2025' AS hindcast_id,
  'fr'                                    AS lang,
  'Paris 2024 Olympic basketball finals (10-11 Aug 2024)'       AS analog_event,
  DATE'2024-08-05'                        AS analog_event_week,
  'NBA Paris Games 2025, Pacers vs Spurs (23 and 25 Jan 2025)'  AS target_event,
  DATE'2025-01-20'                        AS target_event_week,
  8   AS baseline_weeks,
  12  AS horizon_weeks,
  0.5 AS low_multiplier,
  2.0 AS high_multiplier,
  2   AS drift_consecutive_weeks;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW baselines AS
SELECT r.hindcast_id, w.lang, w.topic,
       percentile(CASE WHEN w.week_start >= date_sub(r.analog_event_week, 7 * r.baseline_weeks)
                        AND w.week_start <  r.analog_event_week AND NOT w.is_incomplete
                       THEN w.views END, 0.5) AS analog_baseline,
       percentile(CASE WHEN w.week_start >= date_sub(r.target_event_week, 7 * r.baseline_weeks)
                        AND w.week_start <  r.target_event_week AND NOT w.is_incomplete
                       THEN w.views END, 0.5) AS target_baseline
FROM nba_demand.silver.pageviews_weekly w
JOIN nba_demand.gold.forecast_rule r ON r.lang = w.lang
GROUP BY r.hindcast_id, w.lang, w.topic;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW curve AS
SELECT o.hindcast_id, b.lang, b.topic, t.article, o.week_offset,
       date_add(o.analog_event_week, 7 * o.week_offset) AS analog_week_start,
       date_add(o.target_event_week, 7 * o.week_offset) AS target_week_start,
       b.analog_baseline, b.target_baseline,
       a.views AS analog_views,
       t.views AS actual_views,
       a.views / b.analog_baseline - 1 AS analog_uplift,
       o.low_multiplier, o.high_multiplier, o.drift_consecutive_weeks
FROM (SELECT *, explode(sequence(-baseline_weeks, horizon_weeks)) AS week_offset
      FROM nba_demand.gold.forecast_rule) o
JOIN baselines b ON b.hindcast_id = o.hindcast_id
LEFT JOIN nba_demand.silver.pageviews_weekly a
       ON a.lang = b.lang AND a.topic = b.topic AND a.week_start = date_add(o.analog_event_week, 7 * o.week_offset)
LEFT JOIN nba_demand.silver.pageviews_weekly t
       ON t.lang = b.lang AND t.topic = b.topic AND t.week_start = date_add(o.target_event_week, 7 * o.week_offset);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW banded AS
SELECT *,
       CASE WHEN week_offset >= 0 THEN target_baseline * (1 + analog_uplift) END AS forecast_base,
       CASE WHEN week_offset >= 0 THEN least(target_baseline * (1 + low_multiplier  * analog_uplift),
                                             target_baseline * (1 + high_multiplier * analog_uplift)) END AS forecast_low,
       CASE WHEN week_offset >= 0 THEN greatest(target_baseline * (1 + low_multiplier  * analog_uplift),
                                                target_baseline * (1 + high_multiplier * analog_uplift)) END AS forecast_high,
       actual_views / target_baseline - 1 AS actual_uplift
FROM curve;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW positioned AS
SELECT *,
       coalesce(band_position IN ('below', 'above'), false) AS is_drift_week
FROM (
  SELECT *,
         CASE WHEN week_offset < 0 OR actual_views IS NULL OR forecast_base IS NULL THEN NULL
              WHEN actual_views < forecast_low  THEN 'below'
              WHEN actual_views > forecast_high THEN 'above'
              ELSE 'inside' END AS band_position
  FROM banded
);

In [0]:
%sql
CREATE OR REPLACE TABLE nba_demand.gold.forecast_vs_actual
COMMENT 'Gold: one row per article per week around the target event (8 weeks before to 12 after). Forecast band from the analog vs actual views, drift flags.'
AS
SELECT hindcast_id, lang, topic, article, week_offset, analog_week_start, target_week_start,
       analog_baseline, analog_views, round(analog_uplift, 4) AS analog_uplift,
       target_baseline, actual_views, round(actual_uplift, 4) AS actual_uplift,
       round(forecast_low) AS forecast_low, round(forecast_base) AS forecast_base, round(forecast_high) AS forecast_high,
       actual_views - round(forecast_base) AS error_vs_base,
       round(actual_views / forecast_base - 1, 4) AS error_pct_vs_base,
       band_position, is_drift_week,
       CASE WHEN is_drift_week THEN row_number() OVER (PARTITION BY hindcast_id, topic, is_drift_week, streak_id ORDER BY week_offset) ELSE 0 END AS consecutive_drift_weeks,
       is_drift_week AND row_number() OVER (PARTITION BY hindcast_id, topic, is_drift_week, streak_id ORDER BY week_offset) >= drift_consecutive_weeks AS drift_flag
FROM (
  SELECT *, week_offset - row_number() OVER (PARTITION BY hindcast_id, topic, is_drift_week ORDER BY week_offset) AS streak_id
  FROM positioned
);

In [0]:
%sql
CREATE OR REPLACE TABLE nba_demand.gold.hindcast_summary
COMMENT 'Gold: one row per article. Headline result of the hindcast: analog shape metrics, event-week verdict, drift counts.'
AS
WITH f AS (
  SELECT * FROM nba_demand.gold.forecast_vs_actual WHERE week_offset >= 0
),
peak AS (
  SELECT hindcast_id, topic, max(analog_uplift) AS analog_peak_uplift,
         max_by(week_offset, analog_uplift) AS analog_peak_week_offset
  FROM f WHERE week_offset <= 4
  GROUP BY hindcast_id, topic
),
half_life AS (
  SELECT f.hindcast_id, f.topic, min(f.week_offset - p.analog_peak_week_offset) AS analog_half_life_weeks
  FROM f JOIN peak p ON p.hindcast_id = f.hindcast_id AND p.topic = f.topic
  WHERE f.week_offset > p.analog_peak_week_offset AND f.analog_uplift < p.analog_peak_uplift / 2
  GROUP BY f.hindcast_id, f.topic
)
SELECT f.hindcast_id, f.lang, f.topic, max(f.article) AS article,
       max(f.analog_baseline) AS analog_baseline,
       max(f.target_baseline) AS target_baseline,
       p.analog_peak_uplift, p.analog_peak_week_offset, h.analog_half_life_weeks,
       max(CASE WHEN f.week_offset = 12 THEN f.analog_uplift END) AS analog_persistence_12w,
       max(CASE WHEN f.week_offset = 0 THEN f.actual_uplift END) AS event_week_actual_uplift,
       max(CASE WHEN f.week_offset = 0 THEN round(f.forecast_low  / f.target_baseline - 1, 4) END) AS event_week_band_low_uplift,
       max(CASE WHEN f.week_offset = 0 THEN round(f.forecast_base / f.target_baseline - 1, 4) END) AS event_week_band_base_uplift,
       max(CASE WHEN f.week_offset = 0 THEN round(f.forecast_high / f.target_baseline - 1, 4) END) AS event_week_band_high_uplift,
       max(CASE WHEN f.week_offset = 0 THEN f.band_position END) AS event_week_band_position,
       count_if(f.is_drift_week) AS drift_weeks,
       count(f.band_position) AS weeks_compared,
       min(CASE WHEN f.drift_flag THEN f.target_week_start END) AS drift_flag_first_raised_week
FROM f
JOIN peak p ON p.hindcast_id = f.hindcast_id AND p.topic = f.topic
LEFT JOIN half_life h ON h.hindcast_id = f.hindcast_id AND h.topic = f.topic
GROUP BY f.hindcast_id, f.lang, f.topic, p.analog_peak_uplift, p.analog_peak_week_offset, h.analog_half_life_weeks;

In [0]:
%sql
SELECT * FROM nba_demand.gold.hindcast_summary;

In [0]:
%sql
SELECT topic, week_offset, target_week_start, actual_views, forecast_low, forecast_base, forecast_high,
       band_position, consecutive_drift_weeks, drift_flag
FROM nba_demand.gold.forecast_vs_actual
WHERE week_offset >= 0
ORDER BY topic, week_offset;